In [ ]:
# NOTEBOOK NAME
# PPIvarianceSaver.ipynb
# NOTEBOOK NAME

# # OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

# from pathlib import Path      # used to play with pathnames to save

# # for projecting radar coordinates to lat and lon
# from pyproj import Geod

# # mapping things
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.io import shapereader

# # for adding lat/lon gridlines on plots
# import matplotlib.ticker as mticker
# from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# # SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *

# # for adding a colourful topo base map to the CAPI plots
# from custom_elevation import fetch_srtm, fetch_gebco_local
# from matplotlib.colors import LinearSegmentedColormap
# from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

In [ ]:
# ADD VARIANCE VARIABLES TO THE EXISTING PPI DATA

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 6
RadarDay   = 16
# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '05:05'
LoopEndTime   = '23:55'

# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)
    
    NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi' + '/'
    NetCDFstorageFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi.nc'
    NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile
    
    
    # try to load in the netcdf file and if it doesn't work, just keep going
    try:
        RadarXR = xr.open_dataset(NetCDFstoragePath, decode_timedelta = False) # add the decode_timedelta to shut up a warning
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue
    
    
    GridStats(RadarXR, 'corrected_reflectivity', 3, fill_value=-32.0)                         # add variances and counts for Z
    GridStats(RadarXR, 'corrected_velocity',     3, fill_value=[-300, 0.009155552842798897])  # add variances and counts for velocity 
                                                                    # very specific near 0 value keeps appearing, likely not valid data
    SaveFolder = NetCDFstorageFolder
    SaveFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi_with_variances.nc'
    SavePath = SaveFolder + SaveFile

    RadarXR.to_netcdf(SavePath, encoding={var: {'zlib': True, 'complevel': 3} for var in RadarXR.data_vars})